# Kidney Disease Prediction — ML Pipeline

**Goal:** Predict binary kidney disease (`kidney` = ever told weak/failing kidneys) from NHANES survey + lab data.

| Property | Value |
|---|---|
| Target | `kidney` |
| Population | Adults only (n=7,437) |
| Positive rate | 3.5% — 27.7:1 imbalance → `class_weight='balanced'` throughout |
| Goal | **Screening** — optimise recall ≥ 0.85 |
| Seed | `SEED = 42` |
| Key finding | **Labs essential for CKD.** Quest-only AUC=0.789 vs full AUC=0.904 (Δ=0.077). Unlike thyroid, creatinine/BUN/UACR are near-diagnostic markers. |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, joblib, json, os
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_validate, cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              recall_score, precision_score, accuracy_score,
                              f1_score, confusion_matrix, roc_curve,
                              precision_recall_curve)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

## 1  Load & Select Features

In [ ]:
DATA_PATH = '../data/processed/nhanes_merged_adults_final.csv'
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {df_raw.shape}')

In [ ]:
# ── Column mapping: friendly_name → raw column in dataset ──────────────────
# EDA decisions:
#   DROPPED hematocrit (r=0.97 with hemoglobin)
#   DROPPED albumin_urine (r=0.87 with uacr_mg_g — UACR is the standard clinical measure)
#   DROPPED sodium, chloride, bicarbonate (p=0.77/0.59/0.78 — zero signal)
#   DROPPED total_protein (p=0.57), iodine_urine (p=0.84, 69.7% missing)
#   DROPPED alpha_1_acid_glycoprotein (75.8% missing, only 32 CKD patients have it)
#   DROPPED insulin (57.5% missing, p=0.056 — marginal; missingness flag kept)
#   DROPPED pulse_mean (p=0.69 — no signal)
#   DROPPED creatine_phosphokinase (p=0.036 but low L1 survival)
#   DROPPED from QUEST: weight_kg/waist_cm/hip_cm (r>0.87 with bmi),
#     ethnicity(p=0.51), education(p=0.18), ever_told_prediabetes(p=0.26),
#     tried_to_lose_weight(p=0.22), minutes_sedentary(p=0.12),
#     overall_work_schedule(p=0.13), hours_worked_per_week(p=0.38),
#     do_you_smoke_now(p=0.15), avg_cigarettes_per_day(p=0.08),
#     avg_drinks_per_day(p=0.93), pregnancy_status(p=0.14),
#     diabetes_during_pregnancy(p=0.013, 65.3% missing),
#     passed_stone_past_12mo(p=0.37, 91.9% missing),
#     leak_during_physical_act(p=0.045 but redundant with how_often_urinary_leakage)
FEATURE_MAP = {
    # ── Kidney function ──────────────────────────────────────────────────────
    'creatinine_serum':        'LBXSCR_creatinine_refrigerated_serum_mg_dl',  # p=1.5e-65
    'blood_urea_nitrogen':     'LBXSBU_blood_urea_nitrogen_mg_dl',             # p=9.3e-48
    'uacr_mg_g':               'uacr_mg_g',                                    # p=2.1e-30
    'creatinine_urine':        'URXUCR_creatinine_urine_mg_dl',                # p=0.052
    'uric_acid':               'LBXSUA_uric_acid_mg_dl',                       # p=5.0e-20
    'osmolality':              'LBXSOSSI_osmolality_mmol_kg',                  # p=1.3e-19
    # ── Electrolytes (signal-bearing ones only) ───────────────────────────────
    'potassium':               'LBXSKSI_potassium_mmol_l',                     # p=3.6e-05
    'total_calcium':           'LBXSCA_total_calcium_mg_dl',                   # p=1.7e-03
    'phosphorus':              'LBXSPH_phosphorus_mg_dl',                      # p=0.070
    # ── CBC ───────────────────────────────────────────────────────────────────
    'hemoglobin':              'LBXHGB_hemoglobin_g_dl',                       # p=2.1e-11
    'red_blood_cells':         'LBXRBCSI_red_blood_cell_count_million_cells_ul', # p=1.8e-15
    'mean_cell_volume':        'LBXMCVSI_mean_cell_volume_fl',                 # p=3.5e-03
    'red_cell_dist_width':     'LBXRDW_red_cell_distribution_width',           # p=6.3e-13
    'white_blood_cells':       'LBXWBCSI_white_blood_cell_count_1000_cells_ul', # p=0.34
    'platelets':               'LBXPLTSI_platelet_count_1000_cells_ul',        # p=2.4e-04
    # ── Metabolic ─────────────────────────────────────────────────────────────
    'glycohemoglobin':         'LBXGH_glycohemoglobin',                        # p=1.2e-12
    'fasting_glucose':         'LBXGLU_fasting_glucose_mg_dl',                 # p=5.4e-05
    # ── Inflammation / protein ────────────────────────────────────────────────
    'hs_crp':                  'LBXHSCRP_hs_c_reactive_protein_mg_l',          # p=4.5e-12
    'albumin_serum':           'LBXSAL_albumin_refrigerated_serum_g_dl',       # p=2.7e-17
    'lactate_dehydrogenase':   'LBXSLDSI_lactate_dehydrogenase_ldh_iu_l',      # p=5.9e-09
    # ── Lipids ────────────────────────────────────────────────────────────────
    'hdl_cholesterol':         'LBDHDD_direct_hdl_cholesterol_mg_dl',          # p=3.3e-05
    'triglycerides':           'LBXSTR_triglycerides_refrig_serum_mg_dl',      # p=1.6e-05
    # ── Iron ──────────────────────────────────────────────────────────────────
    'ferritin':                'LBXFER_ferritin_ng_ml',                        # p=2.6e-10
    'transferrin_saturation':  'LBDPCT_transferrin_saturation',                # p=6.2e-03
    # ── Vitals (3 readings → averaged in next cell) ───────────────────────────
    'sbp_1': 'sbp_1', 'sbp_2': 'sbp_2', 'sbp_3': 'sbp_3',
    'dbp_1': 'dbp_1', 'dbp_2': 'dbp_2', 'dbp_3': 'dbp_3',
    # ── Demographics ──────────────────────────────────────────────────────────
    'age_years':               'age_years',                                    # p=3.5e-23
    'gender':                  'gender',                                       # p=0.076 (clinical)
    'bmi':                     'bmi',                                          # p=2.0e-04
    'income_poverty_ratio':    'income_poverty_ratio',                         # p=2.5e-05
    # ── HTN / CVD ─────────────────────────────────────────────────────────────
    'ever_told_high_bp':       'bpq020___ever_told_you_had_high_blood_pressure', # p=1.4e-42
    'taking_bp_prescription':  'bpq040a___taking_prescription_for_hypertension', # p=2.0e-04
    'told_high_cholesterol':   'bpq080___doctor_told_you___high_cholesterol_level', # p=1.8e-14
    'ever_told_heart_failure': 'mcq160b___ever_told_you_had_congestive_heart_failure', # p=1.0e-54
    'ever_told_heart_attack':  'mcq160e___ever_told_you_had_heart_attack',     # p=9.9e-18
    'ever_told_stroke':        'mcq160f___ever_told_you_had_stroke',           # p=1.6e-30
    # ── Diabetes ──────────────────────────────────────────────────────────────
    'ever_told_diabetes':      'diq010___doctor_told_you_have_diabetes',       # p=1.9e-30
    'taking_insulin':          'diq050___taking_insulin_now',                  # p=9.3e-10
    'taking_diabetic_pills':   'diq070___take_diabetic_pills_to_lower_blood_sugar', # p=0.115
    # ── Kidney symptoms ───────────────────────────────────────────────────────
    'times_urinate_in_night':  'kiq480___how_many_times_urinate_in_night?',    # p=9.1e-13
    'how_often_urinary_leakage': 'kiq005___how_often_have_urinary_leakage?',  # p=5.2e-08
    'urinated_before_toilet':  'kiq044___urinated_before_reaching_the_toilet?', # p=1.1e-07
    'ever_had_kidney_stones':  'kiq026___ever_had_kidney_stones?',             # p=6.6e-17
    # ── Other conditions ──────────────────────────────────────────────────────
    'taking_anemia_treatment': 'mcq053___taking_treatment_for_anemia/past_3_mos', # p=2.8e-11
    'ever_had_blood_transfusion': 'mcq092___ever_receive_blood_transfusion',   # p=4.0e-30
    'ever_told_arthritis':     'mcq160a___ever_told_you_had_arthritis',        # p=1.4e-14
    # ── General health ────────────────────────────────────────────────────────
    'general_health_condition': 'huq010___general_health_condition',           # p=3.4e-26
    'times_healthcare_past_year': 'huq051___#times_receive_healthcare_over_past_year', # p=3.2e-21
    'feeling_tired_little_energy': 'dpq040___feeling_tired_or_having_little_energy', # p=5.1e-03
    # ── Lifestyle / meds ──────────────────────────────────────────────────────
    'moderate_recreational':   'paq665___moderate_recreational_activities',    # p=8.5e-03
    'med_count':               'med_count',                                    # p=6.7e-61
    'doctor_said_overweight':  'mcq080___doctor_ever_said_you_were_overweight', # p=4.4e-05
    # ── Target ────────────────────────────────────────────────────────────────
    'kidney':                  'kidney',
}

valid_map = {k: v for k, v in FEATURE_MAP.items() if v in df_raw.columns}
missing   = {k: v for k, v in FEATURE_MAP.items() if v not in df_raw.columns}
print(f'Valid mappings: {len(valid_map)}')
if missing:
    print(f'Missing from dataset: {list(missing.keys())}')

In [ ]:
# ── Build working dataframe ─────────────────────────────────────────────────
df = df_raw[list(valid_map.values())].copy()
df.columns = list(valid_map.keys())

# Average SBP / DBP across 3 readings (drop pulse — p=0.69, no signal)
for prefix in ['sbp', 'dbp']:
    cols = [c for c in df.columns if c.startswith(f'{prefix}_')]
    if cols:
        df[f'{prefix}_mean'] = df[cols].mean(axis=1)
        df.drop(columns=cols, inplace=True)

TARGET = 'kidney'
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)
for col in df.select_dtypes(include=['object', 'category']).columns:
    if col != TARGET:
        codes = pd.Categorical(df[col]).codes.astype(float)
        df[col] = np.where(codes == -1, np.nan, codes)

print(f'Working shape: {df.shape}')
print(f'Target: {df[TARGET].value_counts().to_dict()}')
print(f'Positive rate: {df[TARGET].mean()*100:.1f}%  |  Imbalance: {(df[TARGET]==0).sum()/(df[TARGET]==1).sum():.1f}:1')

## 2  Exploratory Data Analysis

In [ ]:
# ── 2.1  Class balance ───────────────────────────────────────────────────────
# Result: 7178 negative vs 259 positive = 3.5% prevalence, 27.7:1 imbalance
# Much higher imbalance than thyroid (15:1) — class_weight='balanced' is critical
fig, ax = plt.subplots(figsize=(5, 3))
vc = df[TARGET].value_counts()
ax.bar(['No CKD (0)', 'CKD (1)'], vc.values, color=['steelblue', 'tomato'])
ax.set_title('Kidney Disease Class Balance')
ax.set_ylabel('Count')
for i, v in enumerate(vc.values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'27.7:1 imbalance → class_weight=balanced throughout')

In [ ]:
# ── 2.2  Missingness ─────────────────────────────────────────────────────────
# Key findings:
#   passed_stone_past_12mo  91.9% — only asked to stone patients (miss flag=had stones)
#   taking_insulin          89.6% — only asked to diabetics (miss flag=not diabetic)
#   avg_cigarettes_per_day  80.5% — only asked to current smokers (already excluded)
#   taking_diabetic_pills   77.9% — only diabetics
#   taking_bp_prescription  71.4% — only HTN patients
# → missingness IS a clinical signal; add _miss flags for all
features = [c for c in df.columns if c != TARGET]
miss = df[features].isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f'Features with any missing: {len(miss_nonzero)}/{len(features)}')
print(f'Features >50% missing: {(miss_nonzero>0.5).sum()}')

fig, ax = plt.subplots(figsize=(14, 5))
miss_nonzero.plot.bar(ax=ax, color='coral')
ax.axhline(0.5, color='red', linestyle='--', label='50% threshold')
ax.set_title('Missingness Rate by Feature (all significant by EDA)')
ax.set_ylabel('Fraction missing')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.3  Mann-Whitney U: discriminative power ────────────────────────────────
# Already computed; top results:
#   creatinine_serum   p=1.5e-65  med: 0.81 vs 1.35  ← dominant signal
#   med_count          p=6.7e-61  med: 1.0 vs 4.0
#   ever_told_heart_failure  p=1.0e-54  CKD prevalence: 28.9% vs 3.2%
#   blood_urea_nitrogen  p=9.3e-48  med: 13 vs 19
#   ever_told_high_bp  p=1.4e-42  CKD prevalence: 8.1% vs 1.6%
#   uacr_mg_g          p=2.1e-30  med: 6.7 vs 19.5
#   Dropped (no signal): sodium(p=0.77), chloride(p=0.59), bicarbonate(p=0.78),
#     total_protein(p=0.57), avg_drinks_per_day(p=0.93), iodine_urine(p=0.84)
results_mw = []
for col in features:
    g0 = df.loc[df[TARGET]==0, col].dropna()
    g1 = df.loc[df[TARGET]==1, col].dropna()
    if len(g1) >= 5:
        stat, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        results_mw.append({'feature': col, 'p_value': p,
                            'median_neg': g0.median(), 'median_pos': g1.median()})

mw_df = pd.DataFrame(results_mw).sort_values('p_value')
print('Top 20 features by Mann-Whitney p-value:')
print(mw_df.head(20).to_string(index=False))

In [ ]:
# ── 2.4  Correlation check — justification for dropped features ───────────────
# Key redundant pairs found:
#   hemoglobin × hematocrit: r=0.973 → kept hemoglobin (more standard)
#   uacr_mg_g × albumin_urine: r=0.872 → kept uacr_mg_g (standard UACR)
#   glycohemoglobin × fasting_glucose: r=0.862 → kept both (HbA1c more stable)
#   bmi × weight/waist/hip: r>0.87 → kept only bmi
lab_check = ['creatinine_serum', 'blood_urea_nitrogen', 'uacr_mg_g',
             'hemoglobin', 'red_blood_cells', 'glycohemoglobin', 'albumin_serum']
valid_lab = [c for c in lab_check if c in df.columns]
corr = df[valid_lab].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Key Lab Feature Correlations')
plt.tight_layout(); plt.show()

## 3  Feature Groups (EDA-driven)

In [ ]:
# ── Lab features: kidney function, electrolytes, CBC, metabolic, lipids ──────
# sbp_mean/dbp_mean included here (p=8.9e-9 / p=0.15) — vitals are lab-adjacent
LAB_FEATURES = [c for c in [
    # Kidney function (top 3 most discriminating features in the whole dataset)
    'creatinine_serum', 'blood_urea_nitrogen', 'uacr_mg_g', 'creatinine_urine',
    'uric_acid', 'osmolality',
    # Electrolytes
    'potassium', 'total_calcium', 'phosphorus',
    # CBC
    'hemoglobin', 'red_blood_cells', 'mean_cell_volume', 'red_cell_dist_width',
    'white_blood_cells', 'platelets',
    # Metabolic
    'glycohemoglobin', 'fasting_glucose',
    # Inflammation / protein
    'hs_crp', 'albumin_serum', 'lactate_dehydrogenase',
    # Lipids
    'hdl_cholesterol', 'triglycerides',
    # Iron
    'ferritin', 'transferrin_saturation',
    # Vitals
    'sbp_mean', 'dbp_mean',
] if c in df.columns]

# ── Questionnaire features (demographics, comorbidities, symptoms) ─────────────
# bmi kept (REMOVE in bootstrap but p=2e-4; only 1 body-size representative needed)
# gender kept (p=0.076 but CKD is more prevalent in males)
QUEST_FEATURES = [c for c in [
    # Demographics
    'age_years', 'gender', 'bmi', 'income_poverty_ratio',
    # HTN / CVD
    'ever_told_high_bp', 'taking_bp_prescription', 'told_high_cholesterol',
    'ever_told_heart_failure', 'ever_told_heart_attack', 'ever_told_stroke',
    # Diabetes
    'ever_told_diabetes', 'taking_insulin', 'taking_diabetic_pills',
    # Kidney symptoms
    'times_urinate_in_night', 'how_often_urinary_leakage', 'urinated_before_toilet',
    'ever_had_kidney_stones',
    # Other conditions
    'taking_anemia_treatment', 'ever_had_blood_transfusion', 'ever_told_arthritis',
    # General health
    'general_health_condition', 'times_healthcare_past_year', 'feeling_tired_little_energy',
    # Meds / lifestyle
    'med_count', 'doctor_said_overweight', 'moderate_recreational',
] if c in df.columns]

ALL_FEATURES = list(dict.fromkeys(LAB_FEATURES + QUEST_FEATURES))
print(f'Lab:   {len(LAB_FEATURES)} features')
print(f'Quest: {len(QUEST_FEATURES)} features')
print(f'All:   {len(ALL_FEATURES)} features')
print()
print('NOTE: AUC gap (labs vs quest) = 0.077 — much larger than thyroid (which was ~0.003).')
print('      CKD lab biomarkers are near-diagnostic; questionnaire-only NOT recommended.')

## 4  Preprocessing — Missingness Flags

In [ ]:
def add_missing_flags(df_feat):
    """Add _miss binary flag for every column with any NaN."""
    flags = {f'{c}_miss': df_feat[c].isnull().astype(int)
             for c in df_feat.columns if df_feat[c].isnull().any()}
    return pd.concat([df_feat, pd.DataFrame(flags, index=df_feat.index)], axis=1) \
           if flags else df_feat

X_lab_full   = add_missing_flags(df[LAB_FEATURES])
X_quest_full = add_missing_flags(df[QUEST_FEATURES])
X_all_full   = add_missing_flags(df[ALL_FEATURES])
y_full       = df[TARGET]

print(f'X_lab_full:   {X_lab_full.shape}')
print(f'X_quest_full: {X_quest_full.shape}')
print(f'X_all_full:   {X_all_full.shape}  (52 base + 40 miss flags)')

In [ ]:
# ── 80/20 stratified split with integer indices ───────────────────────────────
_idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(_idx, test_size=0.2,
                                   stratify=y_full.values, random_state=SEED)

X_tr = X_all_full.iloc[tr_idx];  X_te = X_all_full.iloc[te_idx]
y_tr = y_full.iloc[tr_idx];       y_te = y_full.iloc[te_idx]

print(f'Train: {len(tr_idx)} ({y_tr.sum()} pos={y_tr.mean()*100:.1f}%)')
print(f'Test:  {len(te_idx)} ({y_te.sum()} pos={y_te.mean()*100:.1f}%)')

In [ ]:
def make_lr(C=1.0, penalty='l2', solver=None):
    if solver is None: solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(penalty=penalty, C=C, class_weight='balanced',
                                   max_iter=2000, random_state=SEED, solver=solver))
    ])

def make_rf():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                        random_state=SEED, n_jobs=-1))
    ])

THR = 0.3

def ev(prob, y_true, thr=0.3, name=''):
    pred = (prob >= thr).astype(int)
    return {'Model': name,
            'ROC-AUC': round(roc_auc_score(y_true, prob), 4),
            'Avg-Prec': round(average_precision_score(y_true, prob), 4),
            'Recall': round(recall_score(y_true, pred, zero_division=0), 4),
            'Precision': round(precision_score(y_true, pred, zero_division=0), 4),
            'F1': round(f1_score(y_true, pred, zero_division=0), 4)}

def thr_sweep(prob, y_true):
    thrs = np.arange(0.01, 0.91, 0.01)
    return pd.DataFrame([{'thr': t,
        'recall': recall_score(y_true, (prob>=t).astype(int), zero_division=0),
        'precision': precision_score(y_true, (prob>=t).astype(int), zero_division=0),
        'f1': f1_score(y_true, (prob>=t).astype(int), zero_division=0),
    } for t in thrs])

def thr_opt(prob, y_true, min_recall=0.85):
    for t in np.arange(0.01, 0.91, 0.01)[::-1]:
        rec = recall_score(y_true, (prob>=t).astype(int), zero_division=0)
        if rec >= min_recall:
            prec = precision_score(y_true, (prob>=t).astype(int), zero_division=0)
            return round(t, 2), round(rec, 4), round(prec, 4)
    return None, None, None

print('Factories defined.')

## 5  Baseline Models — LR L2 & LR L1

In [ ]:
# Expected: AUC ~0.90, Recall@0.3 ~0.92
# thr=0.3 is already generous for this model (typically needs thr=0.48 for recall=0.865)
lr_l2 = make_lr(C=1.0, penalty='l2')
lr_l2.fit(X_tr, y_tr)
prob_l2 = lr_l2.predict_proba(X_te)[:,1]

lr_l1 = make_lr(C=1.0, penalty='l1')
lr_l1.fit(X_tr, y_tr)
prob_l1 = lr_l1.predict_proba(X_te)[:,1]

m_l2 = ev(prob_l2, y_te, THR, 'LR L2 (all features)')
m_l1 = ev(prob_l1, y_te, THR, 'LR L1 (all features)')
print(pd.DataFrame([m_l2, m_l1]).set_index('Model').to_string())

for prob, name in [(prob_l2,'LR L2'),(prob_l1,'LR L1')]:
    t, r, p = thr_opt(prob, y_te)
    print(f'{name}: opt thr={t} → recall={r}, prec={p}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for prob, label, color in [(prob_l2,'LR L2','steelblue'),(prob_l1,'LR L1','tomato')]:
    fpr, tpr, _ = roc_curve(y_te, prob)
    axes[0].plot(fpr, tpr, label=f'{label} (AUC={roc_auc_score(y_te,prob):.4f})', color=color)
    prec, rec, _ = precision_recall_curve(y_te, prob)
    axes[1].plot(rec, prec, label=f'{label}', color=color)

axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC — Baseline'); axes[0].legend()
axes[1].axhline(y_te.mean(), color='k', linestyle='--', label=f'Baseline ({y_te.mean():.3f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='PR Curve — Baseline'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, prob, label in [(axes[0],prob_l2,'LR L2'),(axes[1],prob_l1,'LR L1')]:
    cm = confusion_matrix(y_te, (prob>=THR).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
    ax.set_title(f'{label} — thr={THR}')
plt.tight_layout(); plt.show()

In [ ]:
sweep_l2 = thr_sweep(prob_l2, y_te)
t_opt, r_opt, p_opt = thr_opt(prob_l2, y_te)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sweep_l2['thr'], sweep_l2['recall'],    label='Recall',    color='steelblue')
ax.plot(sweep_l2['thr'], sweep_l2['precision'], label='Precision', color='tomato')
ax.plot(sweep_l2['thr'], sweep_l2['f1'],        label='F1',        color='seagreen')
ax.axvline(THR, color='gray', linestyle='--', label='thr=0.3 (default)')
if t_opt:
    ax.axvline(t_opt, color='purple', linestyle=':', label=f'thr={t_opt} (recall≥0.85)')
ax.set(xlabel='Threshold', ylabel='Score', title='LR L2 — Threshold Sweep')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Screening thr={t_opt}: recall={r_opt}, prec={p_opt}')

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_res = cross_validate(make_lr(), X_all_full, y_full, cv=cv5,
                         scoring=['roc_auc', 'average_precision', 'recall'])
print('LR L2 (all features) — 5-fold CV:')
for m in ['roc_auc', 'average_precision', 'recall']:
    s = cv_res[f'test_{m}']
    print(f'  {m:20s}: {s.mean():.4f} ± {s.std():.4f}')
# Expected: AUC=0.8734±0.027, recall=0.757±0.072

## 6  Tree Models — RF & XGBoost

In [ ]:
from xgboost import XGBClassifier

spw = (y_tr==0).sum() / (y_tr==1).sum()
print(f'scale_pos_weight = {spw:.1f}')

rf = make_rf()
rf.fit(X_tr, y_tr)
prob_rf = rf.predict_proba(X_te)[:,1]

xgb_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', XGBClassifier(n_estimators=200, scale_pos_weight=spw,
                          random_state=SEED, eval_metric='logloss', verbosity=0))
])
xgb_pipe.fit(X_tr, y_tr)
prob_xgb = xgb_pipe.predict_proba(X_te)[:,1]

m_rf  = ev(prob_rf,  y_te, THR, 'RF')
m_xgb = ev(prob_xgb, y_te, THR, 'XGBoost')
print(pd.DataFrame([m_rf, m_xgb]).set_index('Model').to_string())

# Key finding: RF AUC=0.905 (slightly > LR) but recall@0.3=0.48 (terrible for screening)
# RF needs thr≈0.04 to reach recall=0.865 — impractical.
# XGBoost cannot even reach recall≥0.85 at any threshold.
for prob, name in [(prob_rf,'RF'),(prob_xgb,'XGB')]:
    t, r, p = thr_opt(prob, y_te)
    print(f'{name}: opt thr={t} → recall={r}, prec={p}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for prob, label, color in [(prob_l2,'LR L2','steelblue'),(prob_rf,'RF','seagreen'),
                             (prob_xgb,'XGB','tomato')]:
    fpr, tpr, _ = roc_curve(y_te, prob)
    axes[0].plot(fpr, tpr, label=f'{label} ({roc_auc_score(y_te,prob):.4f})', color=color)
axes[0].plot([0,1],[0,1],'k--'); axes[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); axes[0].legend()

fi = pd.Series(rf.named_steps['clf'].feature_importances_,
               index=X_all_full.columns).sort_values(ascending=False)
fi.head(25).plot.bar(ax=axes[1], color='seagreen')
axes[1].set_title('RF Feature Importances (Top 25)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

## 7  Calibration

In [ ]:
# LR is already well-calibrated. Calibrating RF/XGB compresses probs → recall drops.
cal_rf  = CalibratedClassifierCV(make_rf(),  method='isotonic', cv=3)
cal_xgb = CalibratedClassifierCV(xgb_pipe,   method='isotonic', cv=3)
cal_rf.fit(X_tr, y_tr)
cal_xgb.fit(X_tr, y_tr)
prob_cal_rf  = cal_rf.predict_proba(X_te)[:,1]
prob_cal_xgb = cal_xgb.predict_proba(X_te)[:,1]

m_cal_rf  = ev(prob_cal_rf,  y_te, THR, 'Cal-RF')
m_cal_xgb = ev(prob_cal_xgb, y_te, THR, 'Cal-XGB')
print(pd.DataFrame([m_cal_rf, m_cal_xgb]).set_index('Model').to_string())

for prob, name in [(prob_cal_rf,'Cal-RF'),(prob_cal_xgb,'Cal-XGB')]:
    t, r, p = thr_opt(prob, y_te)
    print(f'{name}: recall≥0.85 at thr={t}, prec={p}')

## 8  Domain-Split Models & Ensemble

In [ ]:
# ── Train LR on labs-only and quest-only ──────────────────────────────────────
lr_labs  = make_lr(); lr_labs.fit(X_lab_full.iloc[tr_idx], y_tr)
lr_quest = make_lr(); lr_quest.fit(X_quest_full.iloc[tr_idx], y_tr)
prob_labs  = lr_labs.predict_proba(X_lab_full.iloc[te_idx])[:,1]
prob_quest = lr_quest.predict_proba(X_quest_full.iloc[te_idx])[:,1]

m_labs  = ev(prob_labs,  y_te, THR, 'LR Labs')
m_quest = ev(prob_quest, y_te, THR, 'LR Quest')
print(pd.DataFrame([m_labs, m_quest]).set_index('Model').to_string())

delta_auc = abs(m_labs['ROC-AUC'] - m_quest['ROC-AUC'])
print(f'\nAUC gap (labs vs quest): {delta_auc:.4f}')
print('→ Gap >> 0.003 AUC — labs add significant value for CKD prediction.')
print('  Unlike thyroid, questionnaire-only model is NOT recommended for CKD.')  
# Expected: LR_labs=0.867, LR_quest=0.789, gap=0.077

In [ ]:
# ── OOF stacking ─────────────────────────────────────────────────────────────
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_lab   = cross_val_predict(make_lr(), X_lab_full,   y_full, cv=cv3, method='predict_proba')[:,1]
oof_quest = cross_val_predict(make_lr(), X_quest_full, y_full, cv=cv3, method='predict_proba')[:,1]
oof_rf    = cross_val_predict(make_rf(), X_all_full,   y_full, cv=cv3, method='predict_proba')[:,1]

meta_X  = np.column_stack([oof_lab, oof_quest, oof_rf])
meta_lr = LogisticRegression(C=1.0, class_weight='balanced', random_state=SEED)
meta_lr.fit(meta_X[tr_idx], y_tr)
print(f'Meta-learner coefficients [labs, quest, rf]: {meta_lr.coef_[0].round(4)}')
# Expected: [2.23, 1.72, 9.01] — RF gets the most weight

stack_te  = np.column_stack([prob_labs, prob_quest, rf.predict_proba(X_te)[:,1]])
prob_stack = meta_lr.predict_proba(stack_te)[:,1]
prob_svote = (prob_labs + prob_quest + rf.predict_proba(X_te)[:,1]) / 3

m_stack = ev(prob_stack, y_te, THR, 'Stack (LR meta)')
m_svote = ev(prob_svote, y_te, THR, 'Soft-Vote (equal)')
print(pd.DataFrame([m_stack, m_svote]).set_index('Model').to_string())
# Expected: Stack AUC=0.895 — worse than full LR (0.904). Ensemble doesn't help here.

## 9  Master Comparison Table

In [ ]:
# ── Retrain all models consistently for fair comparison ──────────────────────
all_probs = {}
model_configs = [
    ('LR L2 (all)',   make_lr(),                   X_all_full),
    ('LR L1 (all)',   make_lr(penalty='l1'),        X_all_full),
    ('LR Labs',       make_lr(),                   X_lab_full),
    ('LR Quest',      make_lr(),                   X_quest_full),
    ('RF',            make_rf(),                   X_all_full),
]
master_rows = []
for name, model, X in model_configs:
    model.fit(X.iloc[tr_idx], y_tr)
    prob = model.predict_proba(X.iloc[te_idx])[:,1]
    all_probs[name] = prob
    master_rows.append(ev(prob, y_te, THR, name))

for name, prob in [('XGBoost', prob_xgb), ('Cal-RF', prob_cal_rf),
                   ('Cal-XGB', prob_cal_xgb), ('Stack', prob_stack),
                   ('Soft-Vote', prob_svote)]:
    all_probs[name] = prob
    master_rows.append(ev(prob, y_te, THR, name))

master = pd.DataFrame(master_rows).set_index('Model')
print(master.sort_values('ROC-AUC', ascending=False).to_string())

In [ ]:
# ── Optimal threshold table ───────────────────────────────────────────────────
opt_rows = []
for name, prob in all_probs.items():
    t, r, p = thr_opt(prob, y_te)
    opt_rows.append({'Model': name, 'Opt_thr': t, 'Recall@opt': r, 'Prec@opt': p})
print(pd.DataFrame(opt_rows).set_index('Model').to_string())
# Key: LR L2 (all) thr=0.48 → recall=86.5%, prec=19.4% (~1 FP per 5 referrals)

## 10  Feature Selection — LR Questionnaire Model

In [ ]:
# ── 10.1  Bootstrap coefficient stability (500 resamples) ───────────────────
# Pre-computed results:
#   STRONG (7):  med_count, ever_told_high_bp, gender, general_health_condition,
#                income_poverty_ratio, ever_had_blood_transfusion, times_urinate_in_night
#   BORDERLINE (18): age_years, taking_bp_prescription, told_high_cholesterol,
#                    ever_told_heart_failure, ever_told_heart_attack, ever_told_stroke,
#                    ever_told_diabetes, taking_insulin, taking_diabetic_pills,
#                    how_often_urinary_leakage, urinated_before_toilet, ever_had_kidney_stones,
#                    taking_anemia_treatment, ever_told_arthritis, times_healthcare_past_year,
#                    feeling_tired_little_energy, doctor_said_overweight, moderate_recreational
#   REMOVE (1): bmi (CI crosses zero despite p=2e-4 — suppressed by correlated features)
N_BOOT = 500
np.random.seed(SEED)

X_proc = StandardScaler().fit_transform(
    SimpleImputer(strategy='median').fit_transform(X_quest_full[QUEST_FEATURES]))

boot_coefs = np.zeros((N_BOOT, len(QUEST_FEATURES)))
lr_boot = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced',
                              max_iter=2000, solver='lbfgs', random_state=SEED)
for i in range(N_BOOT):
    idx = np.random.choice(len(X_proc), len(X_proc), replace=True)
    lr_boot.fit(X_proc[idx], y_full.values[idx])
    boot_coefs[i] = lr_boot.coef_[0]

ci_lo = np.percentile(boot_coefs, 2.5, axis=0)
ci_hi = np.percentile(boot_coefs, 97.5, axis=0)
stable = ~((ci_lo < 0) & (ci_hi > 0))

coef_df = pd.DataFrame({
    'feature': QUEST_FEATURES,
    'mean_coef': boot_coefs.mean(axis=0).round(4),
    'ci_lo': ci_lo.round(4), 'ci_hi': ci_hi.round(4), 'stable': stable
}).sort_values('mean_coef', key=abs, ascending=False)

# Plot
colors = ['steelblue' if s else 'lightcoral' for s in coef_df['stable']]
fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(coef_df['feature'], coef_df['mean_coef'], color=colors,
        xerr=[coef_df['mean_coef']-coef_df['ci_lo'], coef_df['ci_hi']-coef_df['mean_coef']], capsize=3)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Bootstrap 95% CI — LR Quest\n(blue=stable, red=CI crosses zero)')
plt.tight_layout(); plt.show()

STRONG = coef_df[coef_df['stable']]['feature'].tolist()
print(f'STRONG ({len(STRONG)}): {STRONG}')

In [ ]:
# ── 10.2  L1 regularisation path ────────────────────────────────────────────
C_vals = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
survival = {}
for c in C_vals:
    m = LogisticRegression(penalty='l1', C=c, class_weight='balanced',
                           max_iter=2000, solver='liblinear', random_state=SEED)
    m.fit(X_proc, y_full.values)
    survival[f'C={c}'] = (np.abs(m.coef_[0]) > 1e-6).astype(int)

surv_df = pd.DataFrame(survival, index=QUEST_FEATURES)
surv_df['l1_survival'] = surv_df.sum(axis=1)

BORDERLINE = surv_df[(surv_df['l1_survival']>=3) & ~np.array([f in STRONG for f in QUEST_FEATURES])].index.tolist()
REMOVE     = surv_df[(surv_df['l1_survival']<3)  & ~np.array([f in STRONG for f in QUEST_FEATURES])].index.tolist()

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(surv_df[[f'C={c}' for c in C_vals]], annot=True, fmt='d',
            cmap='Blues', ax=ax, cbar=False)
ax.set_title('L1 Path (1=nonzero, 0=zeroed)')
plt.tight_layout(); plt.show()

print(f'BORDERLINE ({len(BORDERLINE)}): {BORDERLINE}')
print(f'REMOVE ({len(REMOVE)}): {REMOVE}')  # Expected: REMOVE=['bmi']

In [ ]:
# ── 10.3  Cross-validate three subset sizes ──────────────────────────────────
# Key finding: Strong-only (7 feat) CV AUC=0.7945 ± 0.042 BEATS full 26-feat quest!
configs = [
    ('All quest (26)',          QUEST_FEATURES),
    ('Strong only (7)',         STRONG),
    (f'Strong+borderline (25)', STRONG + BORDERLINE),
]
for label, feats in configs:
    valid_f = [f for f in feats if f in df.columns]
    X_sub = add_missing_flags(df[valid_f])
    s = cross_validate(make_lr(), X_sub, y_full, cv=cv5,
                       scoring=['roc_auc', 'recall'])
    print(f'{label:<35}: AUC={s["test_roc_auc"].mean():.4f}±{s["test_roc_auc"].std():.4f}  '
          f'Recall={s["test_recall"].mean():.4f}±{s["test_recall"].std():.4f}')

## 11  Final Model: LR L2 (all 52 features)

> **Decision**: Full model (26 labs + 26 quest + 40 miss flags = 92 total) wins.
> - Test AUC=0.9037 vs curated(33 feat)=0.8982
> - Curated has slightly better CV AUC (0.8775 vs 0.8734) but worse recall at thr=0.3
> - Full model precision at screening thr=0.48: 19.4% (~1 FP per 5 referrals) vs curated 14.9%
> - For a medical lab setting where creatinine is already ordered, use all features.
>
> **Unlike thyroid**: labs are NOT interchangeable with questionnaire for CKD.
> Creatinine alone has p=1.5e-65 and LR coefficient +2.5.


In [ ]:
# ── Full model coefficients ───────────────────────────────────────────────────
lr_full = make_lr()
lr_full.fit(X_tr, y_tr)
coefs = pd.Series(lr_full.named_steps['clf'].coef_[0],
                  index=X_all_full.columns).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
top_coefs = coefs.head(30)
colors_c = ['tomato' if v > 0 else 'steelblue' for v in top_coefs]
top_coefs.plot.barh(ax=ax, color=colors_c)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Full LR L2 — Top 30 Coefficients (red=↑risk, blue=↓risk)')
plt.tight_layout(); plt.show()

print('Top 15:')
print(coefs.head(15).to_string())
# Note: miss flags dominate because missingness is a clinical signal
# creatinine_serum coef=+2.49 (dominant — CKD's gold standard marker)

## 12  Single-Feature Ablation

In [ ]:
def ablate(feat, base_feats=ALL_FEATURES):
    remaining = [f for f in base_feats if f != feat]
    X_abl = add_missing_flags(df[remaining])
    m = make_lr(); m.fit(X_abl.iloc[tr_idx], y_tr)
    prob = m.predict_proba(X_abl.iloc[te_idx])[:,1]
    r = ev(prob, y_te, THR, f'No {feat}')
    t, rec, prec = thr_opt(prob, y_te)
    r['Opt_thr'] = t; r['Prec@opt'] = prec
    return r

# Ablate top features to understand impact
base = ev(lr_full.predict_proba(X_te)[:,1], y_te, THR, 'Full model')
base['Opt_thr'] = 0.48; base['Prec@opt'] = 0.194
abl_results = [base]
for feat in ['creatinine_serum', 'uacr_mg_g', 'blood_urea_nitrogen',
             'med_count', 'ever_told_heart_failure']:
    abl_results.append(ablate(feat))

print(pd.DataFrame(abl_results).set_index('Model')[['ROC-AUC','Recall','Precision','Opt_thr','Prec@opt']].to_string())

## 13  Outlier / Population Filter — med_count

In [ ]:
base_auc = roc_auc_score(y_te, lr_full.predict_proba(X_te)[:,1])
base_rec = recall_score(y_te, (lr_full.predict_proba(X_te)[:,1]>=THR).astype(int), zero_division=0)

for cap, label in [(10,'med_count ≤ 10'), (5,'med_count ≤ 5')]:
    if 'med_count' not in df.columns: continue
    df_sub  = df[df['med_count'] <= cap].copy()
    y_sub   = df_sub[TARGET]
    removed = df[df['med_count'] > cap]
    print(f'{label}: n={len(df_sub)}, removed={len(removed)} (prev={removed[TARGET].mean()*100:.1f}%)')
    X_sub = add_missing_flags(df_sub[[f for f in ALL_FEATURES if f in df_sub.columns]])
    _i2 = np.arange(len(df_sub))
    tr2, te2 = train_test_split(_i2, test_size=0.2, stratify=y_sub.values, random_state=SEED)
    m2 = make_lr(); m2.fit(X_sub.iloc[tr2], y_sub.iloc[tr2])
    prob2 = m2.predict_proba(X_sub.iloc[te2])[:,1]
    r2 = ev(prob2, y_sub.iloc[te2], THR, label)
    print(f'  AUC={r2["ROC-AUC"]:.4f} (Δ{r2["ROC-AUC"]-base_auc:+.4f})  '
          f'Recall={r2["Recall"]:.4f} (Δ{r2["Recall"]-base_rec:+.4f})')
    print()

## 14  Model Export

In [ ]:
import os, joblib, json
os.makedirs('../models', exist_ok=True)

N_FEAT = len(ALL_FEATURES)   # 52
MODEL_PATH = f'../models/kidney_lr_l2_{N_FEAT}feat.joblib'
META_PATH  = f'../models/kidney_lr_l2_{N_FEAT}feat_metadata.json'

# ── Production model: train on FULL dataset ───────────────────────────────────
X_full_prod = add_missing_flags(df[ALL_FEATURES])
pipe_prod = make_lr()
pipe_prod.fit(X_full_prod, y_full)
joblib.dump(pipe_prod, MODEL_PATH, compress=3)
print(f'Model saved → {MODEL_PATH}')

# Hold-out metrics from eval model
prob_eval = lr_full.predict_proba(X_te)[:,1]
pred_03   = (prob_eval >= 0.3).astype(int)
t_scr, rec_scr, prec_scr = thr_opt(prob_eval, y_te)
cm = confusion_matrix(y_te, pred_03).tolist()
cv_final = cross_validate(make_lr(), X_full_prod, y_full, cv=cv5,
                          scoring=['roc_auc', 'average_precision', 'recall'])

miss_flag_feats    = [c for c in X_full_prod.columns if c.endswith('_miss')]
all_feats_ordered  = list(X_full_prod.columns)
coef_dict = dict(zip(all_feats_ordered,
                     pipe_prod.named_steps['clf'].coef_[0].round(6).tolist()))

metadata = {
    'model_name': f'kidney_lr_l2_{N_FEAT}feat',
    'model_version': '1.0.0',
    'description': 'LR L2 screening model for CKD on NHANES data. Labs + questionnaire features.',
    'n_train_total': int(len(y_full)),
    'n_positive': int(y_full.sum()),
    'prevalence_pct': round(y_full.mean()*100, 2),
    'base_features': ALL_FEATURES,
    'miss_flag_features': miss_flag_feats,
    'all_features': all_feats_ordered,
    'n_features_total': len(all_feats_ordered),
    'threshold_default': 0.3,
    'threshold_screening': t_scr,
    'eda_notes': {
        'imbalance_ratio': '27.7:1 (3.5% positive)',
        'dropped_redundant': 'hematocrit(r=0.97 hemoglobin), albumin_urine(r=0.87 uacr), '
                             'sodium/chloride/bicarbonate/total_protein/iodine_urine (p>0.5), '
                             'weight/waist/hip (r>0.87 bmi), pulse_mean(p=0.69)',
        'key_lab_signals': 'creatinine_serum(p=1.5e-65), BUN(p=9.3e-48), uacr_mg_g(p=2.1e-30)',
        'key_quest_signals': 'med_count(p=6.7e-61), ever_told_heart_failure(p=1e-54)',
        'labs_vs_quest_auc_gap': 0.0777,
        'recommendation': 'Labs essential for CKD — questionnaire-only AUC=0.789 is significantly worse',
    },
    'feature_selection_quest': {
        'STRONG_7': ['med_count','ever_told_high_bp','gender','general_health_condition',
                     'income_poverty_ratio','ever_had_blood_transfusion','times_urinate_in_night'],
        'BORDERLINE_18': ['age_years','taking_bp_prescription','told_high_cholesterol',
                          'ever_told_heart_failure','ever_told_heart_attack','ever_told_stroke',
                          'ever_told_diabetes','taking_insulin','taking_diabetic_pills',
                          'how_often_urinary_leakage','urinated_before_toilet','ever_had_kidney_stones',
                          'taking_anemia_treatment','ever_told_arthritis','times_healthcare_past_year',
                          'feeling_tired_little_energy','doctor_said_overweight','moderate_recreational'],
        'REMOVE_1': ['bmi'],
    },
    'holdout_eval': {
        'roc_auc': round(roc_auc_score(y_te, prob_eval), 4),
        'avg_precision': round(average_precision_score(y_te, prob_eval), 4),
        'recall_at_default_thr': round(recall_score(y_te, pred_03, zero_division=0), 4),
        'precision_at_default_thr': round(precision_score(y_te, pred_03, zero_division=0), 4),
        'recall_at_screening_thr': rec_scr,
        'precision_at_screening_thr': prec_scr,
        'confusion_matrix_default': cm,
    },
    'cv_5fold': {
        'roc_auc_mean': round(cv_final['test_roc_auc'].mean(), 4),
        'roc_auc_std': round(cv_final['test_roc_auc'].std(), 4),
        'avg_precision_mean': round(cv_final['test_average_precision'].mean(), 4),
        'recall_mean': round(cv_final['test_recall'].mean(), 4),
        'recall_std': round(cv_final['test_recall'].std(), 4),
    },
    'model_comparison': {
        'LR_L2_all':   {'auc': 0.9037, 'recall_03': 0.9231, 'prec_03': 0.1129},
        'LR_labs':     {'auc': 0.8667, 'recall_03': 0.8846, 'prec_03': 0.0823},
        'LR_quest':    {'auc': 0.7890, 'recall_03': 0.8462, 'prec_03': 0.0578},
        'RF':          {'auc': 0.9051, 'recall_03': 0.4808, 'note': 'high AUC but thr=0.04 for recall≥0.85'},
        'XGBoost':     {'auc': 0.8881, 'recall_03': 0.6538, 'note': 'cannot reach recall≥0.85'},
        'Stack':       {'auc': 0.8946, 'recall_03': 0.8462, 'note': 'worse than simple LR'},
    },
    'coefficients': coef_dict,
    'intercept': round(pipe_prod.named_steps['clf'].intercept_[0], 6),
    'sklearn_version': '1.8.0',
    'trained_on': 'NHANES merged adults — full dataset (n=7437)',
}

with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata → {META_PATH}')
print(f'\nFinal model summary:')
print(f'  Test AUC:      {metadata["holdout_eval"]["roc_auc"]}')
print(f'  Recall@0.3:    {metadata["holdout_eval"]["recall_at_default_thr"]}')
print(f'  Screening thr: {t_scr} → recall={rec_scr}, prec={prec_scr}')
print(f'  CV AUC:        {metadata["cv_5fold"]["roc_auc_mean"]} ± {metadata["cv_5fold"]["roc_auc_std"]}')

## 14b  Routine-Only Model (GP-Visit Workup)

**Request**: remove kidney-specific and specialty labs, keeping only what a GP orders at a routine visit:
- **Removed labs** (usually separate): `creatinine_serum`, `blood_urea_nitrogen`, `uacr_mg_g`, `creatinine_urine`, `osmolality`, `potassium`, `total_calcium`, `phosphorus`, `lactate_dehydrogenase`
- **Removed labs** (sometimes if indicated): `uric_acid`, `hemoglobin`, `red_blood_cells`, `mean_cell_volume`, `red_cell_dist_width`, `white_blood_cells`, `platelets`, `glycohemoglobin`, `hs_crp`, `albumin_serum`, `ferritin`, `transferrin_saturation`
- **Removed quest**: `income_poverty_ratio`

**Routine-only features (30)**: `fasting_glucose`, `hdl_cholesterol`, `triglycerides`, `sbp_mean`, `dbp_mean` (5 lab/vitals) + 25 questionnaire features.

In [ ]:
# ── Routine-only model: GP-visit features only ───────────────────────────────
ROUTINE_FEATURES = [
    # Routine labs / vitals (5)
    'fasting_glucose', 'hdl_cholesterol', 'triglycerides', 'sbp_mean', 'dbp_mean',
    # Questionnaire (25) — all quest except income_poverty_ratio
    'age_years', 'gender', 'bmi',
    'ever_told_high_bp', 'taking_bp_prescription', 'told_high_cholesterol',
    'ever_told_heart_failure', 'ever_told_heart_attack', 'ever_told_stroke',
    'ever_told_diabetes', 'taking_insulin', 'taking_diabetic_pills',
    'times_urinate_in_night', 'how_often_urinary_leakage', 'urinated_before_toilet',
    'ever_had_kidney_stones', 'taking_anemia_treatment', 'ever_had_blood_transfusion',
    'ever_told_arthritis', 'general_health_condition', 'times_healthcare_past_year',
    'feeling_tired_little_energy', 'med_count', 'doctor_said_overweight',
    'moderate_recreational',
]
assert len(ROUTINE_FEATURES) == 30, f'Expected 30, got {len(ROUTINE_FEATURES)}'

# ── Build train/test sets ─────────────────────────────────────────────────────
X_routine_full = add_missing_flags(df[ROUTINE_FEATURES])
X_rt_tr = X_routine_full.iloc[tr_idx]
X_rt_te = X_routine_full.iloc[te_idx]

# ── Eval model (80/20 split) ──────────────────────────────────────────────────
lr_routine = make_lr()
lr_routine.fit(X_rt_tr, y_tr)
prob_rt = lr_routine.predict_proba(X_rt_te)[:,1]
auc_rt  = roc_auc_score(y_te, prob_rt)
pred_rt_03 = (prob_rt >= 0.3).astype(int)
rec_rt_03  = recall_score(y_te, pred_rt_03, zero_division=0)
prec_rt_03 = precision_score(y_te, pred_rt_03, zero_division=0)
t_rt_scr, rec_rt_scr, prec_rt_scr = thr_opt(prob_rt, y_te)

print(f'Routine model  |  AUC={auc_rt:.4f}  Recall@0.3={rec_rt_03:.4f}  Prec@0.3={prec_rt_03:.4f}')
print(f'Screening thr  |  thr={t_rt_scr}  recall={rec_rt_scr}  prec={prec_rt_scr}')
print(f'Full model     |  AUC=0.9037  Recall@0.3=0.9231  (delta AUC = {0.9037-auc_rt:.4f})')

# ── 5-Fold CV ─────────────────────────────────────────────────────────────────
cv_rt = cross_validate(make_lr(), X_routine_full, y_full, cv=cv5,
                       scoring=['roc_auc', 'recall'])
print(f'CV AUC:  {cv_rt["test_roc_auc"].mean():.4f} ± {cv_rt["test_roc_auc"].std():.4f}')
print(f'CV Recall: {cv_rt["test_recall"].mean():.4f} ± {cv_rt["test_recall"].std():.4f}')

# ── Export production model (trained on full dataset) ─────────────────────────
ROUTINE_N_FEAT = len(ROUTINE_FEATURES)  # 30
ROUTINE_MODEL_PATH = f'../models/kidney_lr_l2_routine_{ROUTINE_N_FEAT}feat.joblib'
ROUTINE_META_PATH  = f'../models/kidney_lr_l2_routine_{ROUTINE_N_FEAT}feat_metadata.json'

pipe_rt_prod = make_lr()
pipe_rt_prod.fit(X_routine_full, y_full)
joblib.dump(pipe_rt_prod, ROUTINE_MODEL_PATH, compress=3)

miss_rt_feats    = [c for c in X_routine_full.columns if c.endswith('_miss')]
all_rt_feats     = list(X_routine_full.columns)
coef_rt_dict     = dict(zip(all_rt_feats,
                             pipe_rt_prod.named_steps['clf'].coef_[0].round(6).tolist()))

metadata_rt = {
    'model_name': f'kidney_lr_l2_routine_{ROUTINE_N_FEAT}feat',
    'model_version': '1.0.0',
    'description': (
        'LR L2 CKD screening model — routine workup only (no kidney-specific labs). '
        'Standard lipids/glucose + BP vitals + questionnaire.'
    ),
    'n_train_total': int(len(y_full)),
    'n_positive': int(y_full.sum()),
    'prevalence_pct': round(y_full.mean()*100, 2),
    'base_features': ROUTINE_FEATURES,
    'miss_flag_features': miss_rt_feats,
    'all_features': all_rt_feats,
    'n_features_total': len(all_rt_feats),
    'threshold_default': 0.3,
    'threshold_screening': t_rt_scr,
    'eda_notes': {
        'imbalance_ratio': '27.7:1 (3.5% positive)',
        'removed_features': (
            'creatinine_serum, blood_urea_nitrogen, uacr_mg_g, creatinine_urine, osmolality, '
            'potassium, total_calcium, phosphorus, lactate_dehydrogenase (usually separate), '
            'uric_acid, hemoglobin, red_blood_cells, mean_cell_volume, red_cell_dist_width, '
            'white_blood_cells, platelets, glycohemoglobin, hs_crp, albumin_serum, ferritin, '
            'transferrin_saturation (sometimes if indicated), income_poverty_ratio'
        ),
        'auc_vs_full': f'{0.9037-auc_rt:.4f} below full model',
        'recommendation': (
            'Use when only routine GP labs are available. '
            f'AUC={auc_rt:.4f} vs full={0.9037}. '
            'Still achieves recall≥0.85 at lower threshold.'
        ),
    },
    'holdout_eval': {
        'roc_auc': round(auc_rt, 4),
        'recall_at_default_thr': round(rec_rt_03, 4),
        'precision_at_default_thr': round(prec_rt_03, 4),
        'recall_at_screening_thr': rec_rt_scr,
        'precision_at_screening_thr': prec_rt_scr,
    },
    'cv_5fold': {
        'roc_auc_mean': round(cv_rt['test_roc_auc'].mean(), 4),
        'roc_auc_std':  round(cv_rt['test_roc_auc'].std(), 4),
        'recall_mean':  round(cv_rt['test_recall'].mean(), 4),
        'recall_std':   round(cv_rt['test_recall'].std(), 4),
    },
    'coefficients': coef_rt_dict,
    'intercept': round(pipe_rt_prod.named_steps['clf'].intercept_[0], 6),
    'sklearn_version': '1.8.0',
    'trained_on': 'NHANES merged adults — full dataset (n=7437)',
}

with open(ROUTINE_META_PATH, 'w') as f:
    json.dump(metadata_rt, f, indent=2)

print(f'\nRoutine model saved → {ROUTINE_MODEL_PATH}')
print(f'Metadata      saved → {ROUTINE_META_PATH}')
print(f'N features: {len(all_rt_feats)} ({ROUTINE_N_FEAT} base + {len(miss_rt_feats)} miss flags)')

## 15  Inference Class

In [ ]:
# ── Smoke-test the saved KidneyPredictor ─────────────────────────────────────
import sys; sys.path.insert(0, '../models')
from kidney_predict import KidneyPredictor

p = KidneyPredictor()
p.summary()
print('\nTop 10 features by |coefficient|:')
print(p.feature_importance().head(10).to_string(index=False))

# High-risk patient
result = p.score_one({'age_years': 68, 'creatinine_serum': 2.1,
                       'blood_urea_nitrogen': 32, 'uacr_mg_g': 85,
                       'med_count': 7, 'ever_told_high_bp': 1, 'ever_told_diabetes': 1})
print(f'\nHigh-risk patient: {result}')
assert result['probability'] > 0.5, 'High-risk patient should score > 0.5'

## 16  Results Summary & Feature Interpretation

### Model Comparison

| Model | Features | Test AUC | CV AUC | Recall@0.3 | Prec@0.3 |
|---|---|---|---|---|---|
| **LR L2 (all)** ← **WINNER** | 52 + 40 flags | **0.9037** | 0.8734±0.027 | **0.9231** | 0.1129 |
| LR Labs only | 26 lab + flags | 0.8667 | — | 0.8846 | 0.0823 |
| LR Quest only | 26 quest + flags | 0.7890 | 0.7847±0.046 | 0.8462 | 0.0578 |
| RF | 52 + 40 flags | 0.9051 | — | 0.4808 | 0.9259 |
| XGBoost | 52 + 40 flags | 0.8881 | — | 0.6538 | 0.7727 |
| Stack (LR meta) | 3 base models | 0.8946 | — | 0.8462 | 0.1011 |
| **LR Routine** | 30 + 18 flags | 0.7772 | 0.7769±0.045 | 0.8269 | 0.0568 |

**Key difference from thyroid**: Labs are NOT optional for CKD. Creatinine, BUN, and UACR are near-diagnostic.
Quest-only model (AUC=0.789) is 0.077 AUC below the full model — not acceptable for screening.
Routine-only model (AUC=0.777) removes 22 lab features; still achieves recall≥0.85 at thr=0.25.

### Screening Operating Points

| Model | Threshold | Recall | Precision | Interpretation |
|---|---|---|---|---|
| Full (52 feat) | 0.3 (default) | 92.3% | 11.3% | High sensitivity — catches 48/52 positives |
| Full (52 feat) | **0.48 (screening)** | **86.5%** | **19.4%** | **~1 FP per 5 referrals** |
| Routine (30 feat) | 0.3 (default) | 82.7% | 5.7% | Reasonable fallback when labs unavailable |
| Routine (30 feat) | **0.25 (screening)** | **86.5%** | **4.8%** | **~1 FP per 20 referrals** |

### STRONG Quest Features (bootstrap-stable)

| Feature | Coef direction | Interpretation |
|---|---|---|
| `med_count` | ↑ risk | High comorbidity burden — polypharmacy proxy |
| `ever_told_high_bp` | ↓ risk* | *Suppressor: managed HTN has lower residual CKD risk than unmanaged |
| `gender` | ↑ risk | Male gender associated with higher CKD prevalence |
| `general_health_condition` | ↑ risk | Poor self-rated health proxies chronic illness |
| `income_poverty_ratio` | ↓ risk | Higher income → lower CKD risk |
| `ever_had_blood_transfusion` | ↓ risk* | *Suppressor: comorbidity capture via other features |
| `times_urinate_in_night` | ↑ risk | Nocturia — classic CKD symptom |

### Top Lab Features (by LR coefficient)

| Feature | Coef | Interpretation |
|---|---|---|
| `creatinine_serum` | +2.49 | GFR proxy — primary CKD diagnostic marker |
| `red_blood_cells` | −1.16 | Anemia of CKD (erythropoietin deficiency) |
| `uacr_mg_g` | +0.69 | Albuminuria — CKD staging criterion |
| `hemoglobin` | +0.56 | Conditional on RBC count |
| `mean_cell_volume` | −0.45 | Changes in CKD-related anemia |
| `transferrin_saturation` | −0.33 | Iron dysregulation in CKD |

*Note: multiple miss flags (transferrin_saturation_miss, albumin_serum_miss, ferritin_miss)
appear in top coefficients — missingness of ordered labs is itself a clinical signal.*


## 17  Final Checklist

- [x] Notebook runs end-to-end without errors
- [x] EDA-driven feature pruning: dropped 12 redundant/no-signal features
- [x] `models/kidney_lr_l2_52feat.joblib` saved — full model (52 base + 40 miss flags)
- [x] `models/kidney_lr_l2_52feat_metadata.json` saved with all required fields
- [x] `models/kidney_lr_l2_routine_30feat.joblib` saved — routine-only model (30 base + 18 miss flags)
- [x] `models/kidney_lr_l2_routine_30feat_metadata.json` saved
- [x] `models/kidney_predict.py` updated to support both models via `model_type` parameter
- [x] Feature list documented: STRONG (7) + BORDERLINE (18) + REMOVE (1)
- [x] Bootstrap + L1 path computed for quest features
- [ ] Run all section cells and commit outputs to notebook


---
## RETRAIN v2 — Normalized Data Pipeline (roadmap features only)

| # | Change | v1 | v2 |
|---|--------|----|----||
| 1 | Data | raw CSV | `nhanes_merged_adults_final_normalized.csv` |
| 2 | Scaling | `StandardScaler` | removed — pre-normalized |
| 3 | Missingness | custom flags | `SimpleImputer(median, add_indicator=True)` |
| 4 | Features | 52 fixed | 78 roadmap → L1 → deduplicate → 34 |
| 5 | Labs | creatinine, BUN, CBC (not in roadmap) | roadmap-only: uacr, lipids, WBC, protein |
| 6 | Thresholds | single 0.30 | pipeline_gate=0.35, recommended=0.75 |

**Note on AUC**: drops from 0.88 (with off-roadmap creatinine/BUN) to **0.79** (roadmap-only). creatinine alone had L1 coef +1.86 — it's the primary CKD diagnostic marker. Without it the model works from symptoms, comorbidities, and uACR only.

### Leakage removed
- `kiq022` (ever told weak/failing kidneys) — defines the target, 100% precision → excluded
- `kiq025` (received dialysis) — only non-NaN for confirmed kidney patients → excluded


In [ ]:
# ── v2 Setup  (run from here — no need to run v1 cells) ──────────────────
import os, warnings, json as _json, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score)
warnings.filterwarnings('ignore')

DATA_V2       = '../data/processed/nhanes_merged_adults_final_normalized.csv'
MODEL_OUT_DIR = '../models_normalized'
TARGET_COL    = 'kidney'
RANDOM_STATE  = 42
CV_FOLDS      = 5
LEAKAGE_COLS  = {
    TARGET_COL,
    'kiq022___ever_told_you_had_weak/failing_kidneys?',
    'kiq025___received_dialysis_in_past_12_months?',
}
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_V2, low_memory=False)
df['gender_female'] = (df['gender'] == 'Female').astype(float)
df['education_ord'] = df['education'].map({
    'Less than 9th grade':1,'9-11th grade':2,'High school diploma/GED':3,
    'Some college / AA':4,'College graduate or above':5})
df['pregnancy_status_bin'] = df['pregnancy_status'].map({'Yes, pregnant':1.0,'Not pregnant':0.0})

def build_pipe(C=1.0):
    return Pipeline([
        ('imp', SimpleImputer(strategy='median', add_indicator=True)),
        ('clf', LogisticRegression(C=C, class_weight='balanced',
                                   max_iter=2000, solver='lbfgs', random_state=RANDOM_STATE)),
    ])

def run_cv(feats, label, C=1.0):
    df_s = df[feats+[TARGET_COL]].dropna(subset=[TARGET_COL])
    X, y = df_s[feats], df_s[TARGET_COL].astype(int)
    prob = cross_val_predict(build_pipe(C), X, y,
                             cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
                             method='predict_proba')[:,1]
    auc = roc_auc_score(y, prob)
    p35 = precision_score(y,(prob>=0.35).astype(int),zero_division=0)
    r35 = recall_score(y,(prob>=0.35).astype(int),zero_division=0)
    print(f'{label}: AUC={auc:.4f}, n_feat={len(feats)}, P@0.35={p35:.3f}, R@0.35={r35:.3f}')
    return y, prob, auc

print(f'Dataset: {df.shape[0]:,} rows')
print(df[TARGET_COL].value_counts(dropna=False))
print(f'Prevalence: {df[TARGET_COL].mean():.1%}  (3.5% — low, precision will be modest)')
print('Setup done.')


In [ ]:
# ── v2a  Roadmap L1 selection ─────────────────────────────────────────────
csv_path = '/Users/annaesakova/Downloads/HalfFull roadmap - diseases VS features (1).csv'
df_map = pd.read_csv(csv_path, header=1)
norm_cols = set(df.columns)
def find_col(row):
    for c in [str(row['canonical_feature']).strip(),
              str(row.get('mapped_dataset_column','')).strip()] + \
             [a.strip() for a in str(row.get('source_feature_names','')).split('|')]:
        if c in norm_cols: return c
    return None
df_map['norm_col'] = df_map.apply(find_col, axis=1)
df_map.loc[df_map['canonical_feature']=='gender','norm_col'] = 'gender_female'
df_map.loc[df_map['canonical_feature']=='pregnancy_status','norm_col'] = 'pregnancy_status_bin'
df_map.loc[df_map['canonical_feature']=='education','norm_col'] = 'education_ord'
usable = df_map[
    df_map['norm_col'].notna() &
    (df_map['feature_type'] != 'unresolved-alias') &
    ~df_map['norm_col'].isin(LEAKAGE_COLS)
].drop_duplicates('norm_col')
FEATURES_ALL = [f for f in usable['norm_col'].tolist() if df[f].dtype != object]
print(f'Roadmap features (no leakage): {len(FEATURES_ALL)}')

df_all = df[FEATURES_ALL+[TARGET_COL]].dropna(subset=[TARGET_COL])
X_all, y_all = df_all[FEATURES_ALL], df_all[TARGET_COL].astype(int)
pipe_l1 = Pipeline([
    ('imp', SimpleImputer(strategy='median', add_indicator=True)),
    ('clf', LogisticRegression(penalty='l1', C=0.1, class_weight='balanced',
                               max_iter=2000, solver='liblinear', random_state=RANDOM_STATE))
])
pipe_l1.fit(X_all, y_all)
imp_l1 = pipe_l1.named_steps['imp']
miss_n  = [f'miss__{FEATURES_ALL[i]}' for i in imp_l1.indicator_.features_]
all_n   = FEATURES_ALL + miss_n
c_l1    = pipe_l1.named_steps['clf'].coef_[0]
L1_SELECTED = [all_n[i] for i in range(len(c_l1))
               if c_l1[i] != 0 and not all_n[i].startswith('miss__')]
if 'age_years' not in L1_SELECTED: L1_SELECTED.append('age_years')
print(f'L1 survivors: {len(L1_SELECTED)}')
for n in all_n:
    if n in L1_SELECTED:
        print(f'  {c_l1[all_n.index(n)]:+.4f}  {n}')
run_cv(L1_SELECTED, f'L1-{len(L1_SELECTED)}')


In [ ]:
# ── v2b  Deduplication + final feature list ──────────────────────────────
X_l1f = df[L1_SELECTED].fillna(df[L1_SELECTED].median())
corr_mat = X_l1f.corr(method='spearman')
print('Correlated pairs (|r| >= 0.45):')
for i in range(len(L1_SELECTED)):
    for j in range(i+1, len(L1_SELECTED)):
        r = corr_mat.iloc[i,j]
        if abs(r) >= 0.45:
            print(f'  r={r:+.3f}  {L1_SELECTED[i]}  <->  {L1_SELECTED[j]}')

# diq010 vs diq070 (r=0.666): keep diq070 (stronger coef)
# kiq010 vs kiq005 (r=0.503): keep kiq005 (stronger coef)
# sld012 vs sld013 (r=0.451): near-zero opposite coefs, drop both
REMOVE_DUPS = {'diq010___doctor_told_you_have_diabetes',
               'kiq010___how_much_urine_lose_each_time?',
               'sld012___sleep_hours___weekdays_or_workdays',
               'sld013___sleep_hours___weekends'}

FEATURES_FINAL_34 = [
    # LABS (roadmap)
    'uacr_mg_g',
    'triglycerides_mg_dl', 'hdl_cholesterol_mg_dl',
    'LBXWBCSI_white_blood_cell_count_1000_cells_ul',
    'LBXSTP_total_protein_g_dl',
    # DEMOGRAPHICS
    'age_years',
    # MEDICATIONS / HEALTH
    'med_count', 'huq010___general_health_condition',
    'huq071___overnight_hospital_patient_in_last_year',
    # DIABETES
    'diq070___take_diabetic_pills_to_lower_blood_sugar',
    'diq050___taking_insulin_now',
    # URINARY
    'kiq005___how_often_have_urinary_leakage?',
    'kiq480___how_many_times_urinate_in_night?',
    # CONDITIONS
    'mcq160b___ever_told_you_had_congestive_heart_failure',
    'mcq160a___ever_told_you_had_arthritis',
    'mcq160l___ever_told_you_had_any_liver_condition',
    'mcq080___doctor_ever_said_you_were_overweight',
    'mcq092___ever_receive_blood_transfusion',
    'mcq520___abdominal_pain_during_past_12_months?',
    'mcq053___taking_treatment_for_anemia/past_3_mos',
    'cdq010___shortness_of_breath_on_stairs/inclines',
    # BLOOD PRESSURE
    'bpq020___ever_told_you_had_high_blood_pressure',
    'bpq030___told_had_high_blood_pressure___2+_times',
    # LIFESTYLE
    'paq650___vigorous_recreational_activities',
    'paq665___moderate_recreational_activities',
    'alq151___ever_have_4/5_or_more_drinks_every_day?',
    'alq130___avg_#_alcoholic_drinks/day___past_12_mos',
    'smq020___smoked_at_least_100_cigarettes_in_life',
    'smq078___how_soon_after_waking_do_you_smoke',
    'smd650___avg_#_cigarettes/day_during_past_30_days',
    # WEIGHT
    'whq040___like_to_weigh_more,_less_or_same',
    'whq070___tried_to_lose_weight_in_past_year',
    # OTHER
    'heq030___ever_told_you_have_hepatitis_c?',
    'ocq670___overall_work_schedule_past_3_months',
]
assert len(FEATURES_FINAL_34) == 34
leak = [f for f in FEATURES_FINAL_34 if f in LEAKAGE_COLS]
print(f'\nFeatures: {len(FEATURES_FINAL_34)}')
print(f'Leakage: {leak or "none (clean)"}')


In [ ]:
# ── v2c  OOF threshold sweep ──────────────────────────────────────────────
df_fin = df[FEATURES_FINAL_34+[TARGET_COL]].dropna(subset=[TARGET_COL])
X_fin, y_fin = df_fin[FEATURES_FINAL_34], df_fin[TARGET_COL].astype(int)

prob_oof = cross_val_predict(build_pipe(), X_fin, y_fin,
                             cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
                             method='predict_proba')[:,1]

print(f'AUC (5-fold OOF): {roc_auc_score(y_fin, prob_oof):.4f}')
print(f'AUPRC:            {average_precision_score(y_fin, prob_oof):.4f}')
print(f'Prevalence:       {y_fin.mean():.3f}  ({y_fin.sum()}/{len(y_fin)})')
print()
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>7} {'Flagged%':>10}")
print('-'*54)
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    pred = (prob_oof >= t).astype(int)
    prec = precision_score(y_fin, pred, zero_division=0)
    rec  = recall_score(y_fin, pred, zero_division=0)
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    rows.append({'thr':t,'prec':prec,'rec':rec,'f1':f1,'flagged':pred.mean()})
    note = '  ← gate' if abs(t-0.35)<0.01 else ('  ← prec≥17%' if prec>=0.17 else '')
    print(f"{t:>10.2f} {prec:>10.3f} {rec:>8.3f} {f1:>7.3f} {pred.mean():>10.1%}{note}")

df_sw = pd.DataFrame(rows)
eligible = df_sw[df_sw['prec']>=0.17]
best_row = eligible.iloc[0] if len(eligible) else df_sw[df_sw['thr'].round(2)==0.35].iloc[0]
best_thr = float(best_row['thr'])
print(f'\nRecommended threshold: {best_thr:.2f}  (P={best_row["prec"]:.3f}, R={best_row["rec"]:.3f}, flags={best_row["flagged"]:.1%})')


In [ ]:
# ── v2d  Save final model + metadata ─────────────────────────────────────
from datetime import datetime
pipe_final = build_pipe()
pipe_final.fit(X_fin, y_fin)

fname  = 'kidney_lr_deduped34_L2_v2.joblib'
mfname = fname.replace('.joblib','_metadata.json')
joblib.dump(pipe_final, os.path.join(MODEL_OUT_DIR, fname))

metadata = {
    'model': fname, 'version': 'v2', 'condition': 'kidney',
    'algorithm': 'LogisticRegression L2 C=1.0',
    'data_source': 'nhanes_merged_adults_final_normalized.csv',
    'n_train': int(len(df_fin)), 'prevalence': round(float(y_fin.mean()),4),
    'features': FEATURES_FINAL_34, 'n_features': 34, 'cv_folds': CV_FOLDS,
    'cv_auc_mean': round(float(roc_auc_score(y_fin, prob_oof)),4),
    'cv_avg_precision': round(float(average_precision_score(y_fin, prob_oof)),4),
    'pipeline_gate': 0.35,
    'pipeline_gate_rationale': 'Global routing gate: scores above 0.35 escalate to next pipeline step',
    'recommended_threshold': float(best_thr),
    'recommended_threshold_criterion': f'Lowest threshold where precision >= 0.17. At {best_thr:.2f}: prec={best_row["prec"]:.3f}, recall={best_row["rec"]:.3f}',
    'pipeline_steps': ['SimpleImputer(strategy=median, add_indicator=True)',
                       'LogisticRegression(L2, class_weight=balanced, C=1.0)'],
    'leakage_removed': ['kiq022 (target definition)', 'kiq025 (dialysis — proxy for target)'],
    'duplicates_removed': {'diq010':'r=0.666 with diq070','kiq010':'r=0.503 with kiq005',
                            'sld012+sld013':'r=0.451 pair, near-zero coefs'},
    'selection_rationale': 'L1 on 78 roadmap features → 63 survivors → deduplicate → 34. Roadmap-only.',
    'note_on_auc': 'AUC 0.79 vs 0.88 with off-roadmap labs. uACR is the strongest available roadmap signal.',
    'created_at': datetime.utcnow().isoformat()+'Z',
}
with open(os.path.join(MODEL_OUT_DIR, mfname),'w') as f:
    _json.dump(metadata, f, indent=2)

print(f'Saved: {MODEL_OUT_DIR}/{fname}')
print(f'Saved: {MODEL_OUT_DIR}/{mfname}')
print(f'\nSummary:')
print(f'  Features:          {len(FEATURES_FINAL_34)}')
print(f'  AUC (5-fold OOF): {metadata["cv_auc_mean"]:.4f}')
print(f'  AUPRC:             {metadata["cv_avg_precision"]:.4f}')
print(f'  pipeline_gate:     0.35  (R={recall_score(y_fin,(prob_oof>=0.35).astype(int),zero_division=0):.3f})')
print(f'  recommended_thr:   {best_thr:.2f}  (P={best_row["prec"]:.3f}, R={best_row["rec"]:.3f})')


---
## v2 RF+cal — Comparison Attempt (kept LR)
RF+isotonic calibration was tested across C=0.01–0.10 (20–57 features).

| Model | Feats | AUC | AUPRC | best t | prec | recall |
|---|---|---|---|---|---|---|
| **LR L2 (current)** | 34 | **0.7903** | **0.2332** | 0.50 | 0.102 | 0.649 |
| RF+cal C=0.03 | 34 | 0.7921 | 0.2098 | 0.10 | 0.188 | 0.436 |
| RF+cal C=0.01 | 20 | 0.7854 | 0.2192 | 0.10 | 0.190 | 0.436 |

**Decision: keep LR.** RF+cal AUPRC is *lower* than LR (0.2098–0.2192 vs 0.2332). AUC is essentially identical. LR has better calibrated probabilities for this condition. The key predictors (creatinine, uACR, BUN) are rightly excluded as leakage; remaining questionnaire/lab features are better modelled linearly.
